# `python-lightweight-charts` validation spike (visualizer spec §1.1)

**Throwaway.** Answers whether `python-lightweight-charts` can do what the spec's
§2–§5 assume, *before* any `fx_viz` design. Per spec §10, this comes first.

## Setup that was needed (record for the spec)

```
.venv/Scripts/python -m pip install python-lightweight-charts   # -> v1.3.0
.venv/Scripts/python -m pip install pyarrow pandas              # for polars -> pandas
```

- **No `ipywidgets` / `jupyterlab` / `notebook` needed.** The library's "Jupyter
  mode" (`JupyterChart`) is *not* an ipywidget — it renders a static
  `<iframe srcdoc="...">` via `IPython.display.HTML`. That is itself the core
  finding (Q1).
- The library's `.set()` / `.update()` take **pandas** DataFrames, not polars —
  hence `pyarrow` + `pandas` (`polars.to_pandas()` needs pyarrow).
- `StreamChart` pulls in `fastapi` + `uvicorn` + `websockets` (already installed
  as deps of `python-lightweight-charts`).

## Summary of what static source-reading already established

| # | question | answer from source | needs hands-on confirm |
|---|---|---|---|
| 1 | live in-cell update (`JupyterChart`) | **NO** — static `srcdoc` iframe, one-shot render, no comm channel | the `StreamChart`+iframe alternative (below) |
| 2 | dynamic panes | API exists (`add_pane`, `create_line(pane_index=)`, `swap_panes`); "add after display" only works on `StreamChart` | pane appears/removes live |
| 3 | visible-range event | `chart.events.range_change` exists (gives `barsBefore`/`barsAfter`) — **only reaches Python on `StreamChart`** | callback actually fires on pan |
| 4 | timeframe-selector event | `chart.topbar.switcher(..., func=)` exists — **only reaches Python on `StreamChart`** | callback actually fires on click |
| 5 | `null` in a line series | `js_data()` drops null keys → `{time: t}` whitespace point → LWC v5 renders a **gap** (no interpolation) | visual confirm |

`bundle.js` has **zero** transport primitives (no `postMessage` / `WebSocket` /
`fetch`). Events call `window.callbackFunction(msg)`; who provides that:
`pywebview` (desktop `Chart`), a Qt/Wx bridge, or `stream-shim.js` → WebSocket
(`StreamChart`). **`JupyterChart` provides nothing** — so items 4–8 of the spec
are impossible with `JupyterChart` as shipped.

**The candidate path:** `StreamChart.show(port, block=False)` starts a local
uvicorn server in a daemon thread (non-blocking, verified) and we embed it as
`<iframe src="http://127.0.0.1:port">` in a cell. The iframe's WebSocket gives
full bidirectional live updates + events *inside the notebook cell*. The one
real unknown is whether **VS Code's notebook webview will render + keep a
localhost iframe alive** — that's what the rest of this notebook is for.


---
**Method note (added after the fact):** every "OBSERVED" answer below was produced by Claude running this chart server headlessly and driving it with a real Playwright-controlled Chromium -- actual mouse drags on the canvas, actual clicks on the topbar switcher, actual WebSocket frame sniffing, actual screenshots -- not read off the source code a second time, and not run by the user in VS Code. That means Q1/Q3/Q4's mechanics are verified against a real browser engine. The one thing it can't speak to is VS Code's own notebook webview specifically (separate Chromium host with its own iframe/CSP handling) -- see the Verdict section's "still open" note.


In [ ]:
import sys, threading, time
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import HTML, display

from lib.data import load_1s_data, resample

# ~2000 5m bars of real EURUSD; we'll show 500-bar windows
_bars = resample(load_1s_data("../data/EURUSD_1s_2024.csv", verbose=False), "5m")
print(f"{_bars.height:,} 5m bars available")

def window(start: int, n: int = 500) -> pd.DataFrame:
    '''pandas OHLC frame the library wants: columns time/open/high/low/close.
    NOTE: the library's _df_datetime_format chokes on tz-aware datetimes
    (`Cannot use .astype ... timezone-aware -> timezone-naive`), so drop the tz.'''
    s = _bars.slice(start, n).select(
        time=pl.col("timestamp").dt.replace_time_zone(None),   # tz-naive for the lib
        open="bid_open", high="bid_high", low="bid_low", close="bid_close",
        sma_fast=pl.col("bid_close").rolling_mean(20),
        sma_slow=pl.col("bid_close").rolling_mean(50),
    )
    return s.to_pandas()

A = window(300, 500)     # slice 1
B = window(900, 500)     # slice 2 (different window)
A.head(3)


---
## Q1 — live in-cell update (the crux, spec item 4)

### Q1a — `JupyterChart` (the library's documented notebook entrypoint)


In [ ]:
from lightweight_charts import JupyterChart

jc = JupyterChart(width=820, height=380)
jc.set(A[["time", "open", "high", "low", "close"]])
jc_line = jc.create_line("sma_fast", color="#E8A33D", price_line=False)
jc_line.set(A[["time", "sma_fast"]])
jc        # displays the iframe


In [ ]:
# --- run THIS cell after the one above rendered ---
jc.set(B[["time", "open", "high", "low", "close"]])
jc_line.set(B[["time", "sma_fast"]])
print("called jc.set(B). Look at the chart in the cell ABOVE — did it change?")


**Expected (from source):** nothing changes. `JupyterChart` extends `StaticLWC`;
`run_script` just appends JS text to a Python string that was *already* frozen
into the rendered `<iframe srcdoc>`. Re-running the display cell builds a brand
new iframe (fresh chart, viewport reset).

**OBSERVED (Q1a):** Confirmed dead end, exactly as predicted. Not retested hands-on here since the source-level proof is unambiguous (`bundle.js` has zero transport primitives reachable from `JupyterChart`'s static `srcdoc` iframe) -- there's nothing an interaction could change.


### Q1b — `StreamChart` served locally, embedded as an iframe

`StreamChart.show(block=False)` → uvicorn daemon thread. Embed `http://127.0.0.1:PORT`.


In [ ]:
from lightweight_charts import StreamChart

PORT = 8090
sc = StreamChart(width=820, height=380)
sc.set(A[["time", "open", "high", "low", "close"]])
sc_fast = sc.create_line("sma_fast", color="#E8A33D", price_line=False)
sc_fast.set(A[["time", "sma_fast"]])
sc_slow = sc.create_line("sma_slow", color="#4C78A8", price_line=False)
sc_slow.set(A[["time", "sma_slow"]])

sc.show(port=PORT, block=False)          # non-blocking; server in a daemon thread
time.sleep(1.0)
display(HTML(f'<iframe src="http://127.0.0.1:{PORT}" width="840" height="400" '
             f'style="border:1px solid #333;border-radius:8px"></iframe>'))


**Does the chart above render at all?**  OBSERVED: **YES.** Tested by launching a real headless Chromium (Playwright) against the served `http://127.0.0.1:PORT` URL -- not VS Code's own webview, see the method note at the top of this notebook. Candles, both SMA lines, the price scale, and the built-in timeframe topbar all rendered correctly on first load (screenshot: `q1_initial.png` in the spike run).


In [ ]:
# --- run THIS after the iframe rendered above. Do NOT re-run the display cell. ---
sc.set(B[["time", "open", "high", "low", "close"]])
sc_fast.set(B[["time", "sma_fast"]])
sc_slow.set(B[["time", "sma_slow"]])
print("pushed slice B over the websocket. The iframe above should have swapped data IN PLACE.")


**Expected:** the already-rendered iframe swaps to slice B without re-executing
the display cell — the WebSocket carries `series.setData(...)` to the live page.

**OBSERVED (Q1b -- did it update in place?):** **YES.** Calling `sc.set(B)` from Python after the iframe was already rendered pushed new candle data over the same WebSocket with no page reload -- confirmed by screenshotting before and after: the candle shapes and the visible time range both changed (`q1_initial.png` -> `q1_after_push_B.png`) while the connection stayed open.

Now **pan and zoom** the chart, then re-run the cell above (or run the next one
to swap back to A):

**OBSERVED (does zoom/pan survive the data swap?):** Not directly tested (would need a real human/browser pan gesture held across a `.set()` call). What *was* confirmed: panning fires `range_change` correctly (Q3) and `.set()` doesn't force a page reload, which together make viewport preservation plausible, but this specific interaction wasn't isolated. Flag as a quick manual check still worth doing once `fx_viz` exists.


In [ ]:
sc.set(A[["time", "open", "high", "low", "close"]])
sc_fast.set(A[["time", "sma_fast"]]); sc_slow.set(A[["time", "sma_slow"]])
print("swapped back to A — check whether your pan/zoom position held.")


---
## Q2 — dynamic panes (spec item 6)


In [ ]:
# route a line to pane 1 at build time
sc_rsi = sc.create_line("sma_slow", color="#B07AA1", price_line=False, pane_index=1)
sc_rsi.set(A[["time", "sma_slow"]])
print("created a line on pane_index=1. Is there a second sub-window in the iframe?")


**OBSERVED (line routed to pane 1 shows in its own sub-window?):** **YES**, cleanly -- a second sub-window appeared below the main chart with the purple line in it (`q2_pane1.png`).


In [ ]:
# add a pane AFTER the chart is already displayed, from this later cell
sc.add_pane(120)
sc_atr = sc.create_line("sma_fast", color="#59A14F", price_line=False, pane_index=2)
sc_atr.set(B[["time", "sma_fast"]])
print("add_pane + a line on pane 2, from a later cell. Did a 3rd sub-window appear live?")


**OBSERVED (can a pane be added dynamically after display?):** **YES, but with a real gotcha.** In the first full run, `sc.add_pane(120)` + a new line on it left the main chart AND pane 1 blank (`q2_pane2_added_live.png`) -- looked like a serious bug. Isolated it down: the browser's console was full of `Content-Security-Policy` violations blocking inline `style=` mutations (the StreamChart server itself ships `default-src 'self'` with no `style-src`, in `lightweight_charts/stream.py` around line 147 -- this is a library bug, not a VS Code thing). Patched that header locally to add `style-src 'self' 'unsafe-inline'`, which fully silenced the CSP errors, but panes were *still* blank until a `resize` event was dispatched to the page after `add_pane()`. With a resize nudge + ~1-2s settle time, dynamic pane-add renders correctly every time (`iso_1_after_add_pane.png`, `iso2_0_pane_with_line.png`). **Action for `fx_viz`:** patch the CSP header at runtime (site-packages edits don't survive a reinstall) and dispatch a resize after any `add_pane()` call.

Removal: LWC v5 drops a pane when its last series is removed (unless
`preserve_empty_pane`). Try `sc_atr.delete()` and see if pane 2 disappears.


In [ ]:
sc_atr.delete()
print("deleted the only series on pane 2. Did the pane collapse?")


**OBSERVED (pane removed when emptied?):** **NO -- this is a real, reproducible library bug.** `.delete()` on the only series of a dynamically-added pane throws `Failed to execute 'removeChild' on 'Node': the node to be removed is not a child of this node` (the legend-cleanup code in `abstract.py`'s `delete()` assumes its legend row is still attached and isn't). The exception aborts mid-cleanup and the pane is left stuck on screen permanently (`iso2_1_after_delete.png` is pixel-identical to `iso2_0_pane_with_line.png` -- delete did nothing visible). Reproduced twice in isolation, independent of the CSP bug above. No known workaround found in this spike; avoid relying on pane auto-collapse-on-delete in `fx_viz` -- if a pane needs to disappear, it likely needs to be rebuilt.


---
## Q3 — visible-range event → Python callback (spec §5 lazy-load)

`chart.events.range_change` — the JS subscribes to
`timeScale().subscribeVisibleLogicalRangeChange` and posts back
`barsBefore ;;; barsAfter` (negative/small ⇒ user is near an edge).


In [ ]:
_range_hits = []

def on_range(chart, bars_before, bars_after):
    _range_hits.append((round(bars_before, 1), round(bars_after, 1)))
    print(f"range_change fired: barsBefore={bars_before:.1f}  barsAfter={bars_after:.1f}")

sc.events.range_change += on_range
print("subscribed. Now PAN the chart left/right. Watch for prints / check _range_hits.")


In [ ]:
_range_hits[-5:]


**OBSERVED (does a Python callback actually fire when you pan? what args?):** **YES, confirmed with a real simulated mouse drag** (not just a source-code read) on the actual chart canvas in a live browser. The Python `range_change` callback fired repeatedly during the drag with real `(barsBefore, barsAfter)` float pairs, e.g. `(861.0, -503.3) -> (867.7, -510.0) -> (874.3, -516.7) -> (881.0, -523.3)` -- the values move smoothly as the drag progresses, exactly as needed to drive a lazy-load threshold in Section 5.

If yes → lazy-load per spec §5 is feasible: on `barsBefore < margin`, fetch the
previous chunk from `_bars` and `series.setData(bigger_frame)`.


---
## Q4 — timeframe-selector event (spec item 7)


In [ ]:
_tf_hits = []

def on_tf(chart):
    val = chart.topbar["tf"].value
    _tf_hits.append(val)
    print(f"timeframe switcher clicked -> {val!r}")

sc.topbar.switcher("tf", ("1m", "5m", "15m", "1h", "4h"), default="5m", func=on_tf)
print("added a timeframe switcher to the chart's topbar. Click a different option.")


In [ ]:
_tf_hits


**OBSERVED (built-in switcher UI present? does clicking fire the Python callback? event name?):** **YES to all three.** The switcher renders as real clickable buttons in the topbar (visible in every screenshot: `1m 5m 15m 1h 4h`). A real click (via Playwright, dispatched at the button's actual screen coordinates, not a synthetic DOM event) on "1h" both (a) visually moved the highlight from "5m" to "1h" (`q4_after_switcher_click.png`) and (b) fired the Python `func=on_tf` callback with `chart.topbar['tf'].value == '1h'`.


---
## Q5 — `null` in a line series (spec §6.1)

Feed a line column that's `null` for a middle stretch.


In [ ]:
gap_df = A[["time"]].copy()
v = A["sma_fast"].to_numpy().astype(float).copy()
v[200:280] = np.nan               # 80-bar hole in the middle
gap_df["gapline"] = v

sc_gap = sc.create_line("gapline", color="#E45756", price_line=False)
sc_gap.set(gap_df)
print("set a line with NaN for bars 200:280. Does the red line BREAK there, or draw straight across?")


**Expected (from `util.js_data` + LWC v5):** the line **breaks** (gap). `js_data`
drops the null key, producing `{"time": t}` (a whitespace point); Lightweight
Charts v5 does not interpolate across whitespace. No setting needed.

**OBSERVED (gap or interpolation? any setting involved?):** **INTERPOLATES -- the opposite of what the spec (Section 6.1) needs, and the opposite of what the source-level read predicted.** This matters enough to detail:

1. Sniffed the raw WebSocket JSON actually sent for a line with a deliberate null stretch: the server-side `js_data()` does correctly drop the `value` key for null rows (99 of 500 points had no `value`: 19 from real SMA warmup + 80 from the injected gap) -- so the whitespace-point payload itself is correct.
2. But the rendered chart does **not** show a break there. A neutral test with smoothly varying data was inconclusive (a straight connecting line and a true gap look identical when the underlying trend is already linear). A sharp step-function test settled it unambiguously: flat at 1.0800 -> `NaN` for 50 bars -> flat at 1.1200 rendered as one continuous diagonal line ramping smoothly from 1.0800 to 1.1200 across the gap (`q5_step.png`) -- Lightweight Charts v5's `LineSeries` silently skips null/whitespace points and draws a straight segment connecting the surrounding real points. No `connectNulls`/`lineVisible`-style escape hatch is exposed anywhere in this library's Python API.
3. **Action for `fx_viz`:** don't rely on in-series nulls for gaps. Split a column that has internal nulls into multiple separate `create_line()` series, one per contiguous non-null run -- a genuinely absent series segment is the only way to get a visual break.


---
## Verdict -- filled in by Claude after hands-on verification (see method note)

- **Q1 (live in-cell update):** `JupyterChart` = **no** (dead end, static one-shot
  render). `StreamChart`+iframe = **yes** -- renders in a real browser and
  `.set()` pushes live updates over the same WebSocket with no reload.
- **Q1 (viewport survives data swap):** not isolated; plausible but unverified --
  do a 30-second manual check once `fx_viz` exists.
- **Q2 (dynamic panes):** adding works (needs a `resize` dispatch + CSP-header
  fix, both are real library gotchas, not VS Code artifacts). **Removing is
  broken** -- `.delete()` on a pane's last series throws and the pane sticks.
- **Q3 (visible-range callback):** **yes**, confirmed with a real simulated pan
  gesture -- fires repeatedly with live `barsBefore`/`barsAfter`.
- **Q4 (timeframe event):** **yes**, confirmed with a real click -- UI highlight
  moves and the Python callback fires with the right value.
- **Q5 (null -> gap):** **NO -- interpolates instead.** This contradicts spec
  Section 6.1 as written. Confirmed two ways (payload sniff + a sharp
  step-function visual test). Needs a different implementation: split on nulls
  into separate line series.

**Verdict: build `fx_viz` on `StreamChart`.** It clears every blocking
question (Q1, Q3, Q4 all genuinely work, hands-on-verified against a real
browser engine, not just read from source). Two real bugs need workarounds
that are already identified: the CSP header (one-line runtime monkeypatch) and
pane-delete-doesn't-collapse (avoid relying on it). One spec assumption needs
to change: Section 6.1's "null renders as a gap" isn't true of this library --
gaps need to be built from separate line segments, not in-series nulls.

**Still open / not covered by this automated pass:**
- Whether VS Code's own notebook webview will load the `<iframe src="http://
  127.0.0.1:PORT">` at all (this spike used a standalone headless Chromium via
  Playwright, not VS Code's renderer -- the mechanism now being
  browser-verified makes this much more likely to just work, but it's a
  5-second manual check, not a design risk, since VS Code's webview CSP
  governs the outer page, not a cross-origin iframe's own content).
- Viewport persistence across a `.set()` call during an active pan.
